# NL2BI / Spider2 — Colab session setup for agent

**Цель ноутбука:** один раз подготовить Colab runtime так, чтобы внешний агент мог работать без отвлечения на инфраструктуру.

## Порядок запуска

1. `00_SESSION_CONFIG` — базовые пути и режимы проверки.
2. `01_MOUNT_DRIVE_AND_DIRS` — Google Drive + рабочие директории.
3. `02_INSTALL_CORE_PACKAGES` — зависимости.
4. `03_SECRETS_AND_AUTH` — HF / BigQuery / Snowflake credentials без хардкода секретов.
5. `04_BIGQUERY_PROBE` — проверка BigQuery.
6. `05_SNOWFLAKE_PROBE` — проверка Snowflake, если credentials заданы.
7. `06_GPU_AND_MODEL_CACHE_PROBE` — GPU/CUDA/HF cache.
8. `07_AGENT_BRIDGE_SETUP` — bridge для агента.
9. `08_FULL_AGENT_READINESS_CHECK` — итоговая проверка готовности.

## Важно по безопасности

Не вставляй service account JSON, HF token, Snowflake password прямо в notebook. Используй:
- файл на Drive: `/content/drive/MyDrive/diploma_plan_sql/secrets/spider2_bq_sa.json`;
- переменные окружения;
- Colab Secrets / userdata.


## 00 — Session config

Здесь задаются пути и режимы проверки. Обычно менять нужно только `PROJECT_ROOT` и флаги `REQUIRE_*`.


In [2]:
# 00_SESSION_CONFIG
from __future__ import annotations

import os
from pathlib import Path

MARKER = "00_SESSION_CONFIG"
print(MARKER)

PROJECT_ROOT = Path(os.environ.get("DIPLOMA_PROJECT_ROOT", "/content/drive/MyDrive/diploma_plan_sql"))
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
SECRETS_DIR = PROJECT_ROOT / "secrets"
BRIDGE_URL_FILE = PROJECT_ROOT / ".bridge_url"

# Credentials / secrets. Do not hardcode secrets in the notebook.
BQ_SA_PATH = Path(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", str(SECRETS_DIR / "spider2_bq_sa.json")))
SNOWFLAKE_SECRET_JSON = Path(os.environ.get("SNOWFLAKE_SECRET_JSON", str(SECRETS_DIR / "snowflake.json")))

# Readiness policy. Set to False only if that lane is intentionally deferred.
REQUIRE_HF_TOKEN = True
REQUIRE_BQ = True
REQUIRE_SNOWFLAKE = True
REQUIRE_GPU = True
REQUIRE_SPIDER2_DATA = True
REQUIRE_BRIDGE = True

# Optional; DBT often runs on a separate server, so it is warning-only by default.
CHECK_DBT_REMOTE = False
DBT_REMOTE = os.environ.get("DBT_REMOTE", "denis@103.54.18.91")
DBT_REMOTE_PROJECT = os.environ.get("DBT_REMOTE_PROJECT", "/home/denis/dbt")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("SECRETS_DIR:", SECRETS_DIR)
print("BQ_SA_PATH exists now:", BQ_SA_PATH.exists())
print("SNOWFLAKE_SECRET_JSON exists now:", SNOWFLAKE_SECRET_JSON.exists())


00_SESSION_CONFIG
PROJECT_ROOT: /content/drive/MyDrive/diploma_plan_sql
OUTPUTS_DIR: /content/drive/MyDrive/diploma_plan_sql/outputs
SECRETS_DIR: /content/drive/MyDrive/diploma_plan_sql/secrets
BQ_SA_PATH exists now: False
SNOWFLAKE_SECRET_JSON exists now: False


## 01 — Drive mount and folders

Монтирует Drive, создаёт базовые папки, проверяет, что рабочий корень доступен на запись.


In [3]:
# 01_MOUNT_DRIVE_AND_DIRS (v2 — robust mount with pre-clear + verification)
# Why this exists: a previous version called drive.mount(force_remount=False)
# and swallowed exceptions. If /content/drive already had local-fs files
# from a prior session, mount silently failed and subsequent mkdir created
# directories on the LOCAL filesystem that shadow the would-be Drive mount,
# making real Drive content invisible. v2 detects that case explicitly,
# clears the stale shadow, mounts with verification, and refuses to proceed
# silently if mount fails.
from __future__ import annotations

import os
import shutil
import datetime as dt
import json
from pathlib import Path

MARKER = "01_MOUNT_DRIVE_AND_DIRS"
print(MARKER)

DRIVE_MNT = Path("/content/drive")


def _is_mount(p: Path) -> bool:
    try:
        return os.path.ismount(str(p))
    except Exception:
        return False


# 1. Pre-mount: if /content/drive is a regular dir (not a mount), clear it.
if DRIVE_MNT.exists() and not _is_mount(DRIVE_MNT):
    n_items = sum(1 for _ in DRIVE_MNT.iterdir())
    print(f"DRIVE_MNT_PRECLEAR: {DRIVE_MNT} exists but is NOT a mount ({n_items} items); clearing")
    try:
        shutil.rmtree(DRIVE_MNT)
        print("DRIVE_MNT_PRECLEAR_OK")
    except Exception as exc:
        print("DRIVE_MNT_PRECLEAR_FAIL:", type(exc).__name__, str(exc)[:300])

# 2. Mount Drive — force_remount=True so a stuck previous mount also gets refreshed.
try:
    from google.colab import drive
    drive.mount(str(DRIVE_MNT), force_remount=True)
    print("DRIVE_MOUNT_OK")
except Exception as exc:
    print("DRIVE_MOUNT_FAIL:", type(exc).__name__, str(exc)[:500])
    raise

# 3. Verify mount actually took: /content/drive must be a mount point AND
# /content/drive/MyDrive must exist. If not, fail hard rather than silently
# creating local-fs shadow dirs.
if not _is_mount(DRIVE_MNT):
    raise RuntimeError(f"{DRIVE_MNT} did not become a mount point after drive.mount; aborting")
if not (DRIVE_MNT / "MyDrive").is_dir():
    raise RuntimeError(f"{DRIVE_MNT}/MyDrive missing after mount — Drive structure unexpected")
print("DRIVE_VERIFY_OK is_mount=True MyDrive=True")

# 4. PROJECT_ROOT and OUTPUTS_DIR come from cell 00_SESSION_CONFIG.
# They MUST already point inside the now-mounted Drive.
if not str(PROJECT_ROOT).startswith(str(DRIVE_MNT)):
    print(f"WARN: PROJECT_ROOT={PROJECT_ROOT} is NOT under {DRIVE_MNT} — proceeding but check 00_SESSION_CONFIG")

# 5. PROJECT_ROOT existence check (don't auto-create on Drive blindly — verify first)
if PROJECT_ROOT.exists():
    n_items = sum(1 for _ in PROJECT_ROOT.iterdir())
    print(f"PROJECT_ROOT exists on Drive: {PROJECT_ROOT}  items={n_items}")
    if n_items == 0:
        print(f"WARN: PROJECT_ROOT is empty — verify the path matches your Drive layout. "
              f"Browse to it in Drive UI; if files are at a different path, set "
              f"DIPLOMA_PROJECT_ROOT env var in cell 00 before re-running.")
else:
    print(f"PROJECT_ROOT not yet present on Drive: {PROJECT_ROOT}  — creating")
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# 6. Ensure required sub-dirs (idempotent on Drive)
for sub in ["outputs", "outputs/logs", "outputs/tables", "outputs/predictions",
              "outputs/plots", "secrets", "external_benchmarks"]:
    (PROJECT_ROOT / sub).mkdir(parents=True, exist_ok=True)

# 7. Write probe and READ BACK to verify the file actually went to Drive.
probe_path = OUTPUTS_DIR / "logs" / "colab_session_write_probe.json"
probe = {
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "drive_is_mount": _is_mount(DRIVE_MNT),
}
probe_path.write_text(json.dumps(probe, ensure_ascii=False, indent=2) + "\n",
                          encoding="utf-8")
back = json.loads(probe_path.read_text(encoding="utf-8"))
if back.get("project_root") != str(PROJECT_ROOT):
    raise RuntimeError(f"WRITE_PROBE integrity FAIL: read-back mismatch {back}")
if not back.get("drive_is_mount"):
    raise RuntimeError("WRITE_PROBE integrity FAIL: drive_is_mount=False at write time")
print("WRITE_PROBE_OK", probe_path, probe_path.stat().st_size, "bytes")

# 8. Re-check the secret files now that Drive is properly mounted.
print(f"BQ_SA_PATH exists now: {BQ_SA_PATH.exists()}")
print(f"SNOWFLAKE_SECRET_JSON exists now: {SNOWFLAKE_SECRET_JSON.exists()}")


Mounted at /content/drive
DRIVE_MOUNT_OK
DRIVE_VERIFY_OK is_mount=True MyDrive=True
PROJECT_ROOT exists on Drive: /content/drive/MyDrive/diploma_plan_sql  items=13
WRITE_PROBE_OK /content/drive/MyDrive/diploma_plan_sql/outputs/logs/colab_session_write_probe.json 134 bytes
BQ_SA_PATH exists now: True
SNOWFLAKE_SECRET_JSON exists now: True


## 02 — Install/import core packages

Ставит только инфраструктурные зависимости. Тяжёлые model-specific установки лучше делать в runner-скриптах агента.


In [4]:
# 02_INSTALL_CORE_PACKAGES
from __future__ import annotations

import importlib
import subprocess
import sys

MARKER = "02_INSTALL_CORE_PACKAGES"
print(MARKER)

PACKAGES = [
    "flask",
    "requests",
    "huggingface_hub",
    "google-cloud-bigquery",
    "google-auth",
    "snowflake-connector-python",
    "sqlglot",
    "jsonschema",
    "func_timeout",
    "rank_bm25",
]

IMPORTS = {
    "flask": "flask",
    "requests": "requests",
    "huggingface_hub": "huggingface_hub",
    "google-cloud-bigquery": "google.cloud.bigquery",
    "google-auth": "google.oauth2.service_account",
    "snowflake-connector-python": "snowflake.connector",
    "sqlglot": "sqlglot",
    "jsonschema": "jsonschema",
    "func_timeout": "func_timeout",
    "rank_bm25": "rank_bm25",
}

missing = []
for pkg, mod in IMPORTS.items():
    try:
        importlib.import_module(mod)
    except Exception:
        missing.append(pkg)

if missing:
    print("INSTALLING", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("ALL_PACKAGES_ALREADY_AVAILABLE")

for pkg, mod in IMPORTS.items():
    importlib.import_module(mod)
    print("IMPORT_OK", mod)


02_INSTALL_CORE_PACKAGES
INSTALLING ['snowflake-connector-python', 'func_timeout', 'rank_bm25']
IMPORT_OK flask
IMPORT_OK requests
IMPORT_OK huggingface_hub
IMPORT_OK google.cloud.bigquery
IMPORT_OK google.oauth2.service_account
IMPORT_OK snowflake.connector
IMPORT_OK sqlglot
IMPORT_OK jsonschema
IMPORT_OK func_timeout
IMPORT_OK rank_bm25


## 03 — Secrets and auth

Проверяет токены без вывода секретов. Если HF token не задан, попросит ввести его через `getpass`.

Для BigQuery положи service account JSON сюда:

`/content/drive/MyDrive/diploma_plan_sql/secrets/spider2_bq_sa.json`

Для Snowflake можно использовать JSON:

`/content/drive/MyDrive/diploma_plan_sql/secrets/snowflake.json`

Ожидаемые ключи Snowflake JSON: `account`, `user`, `password`, `role`, `warehouse`, `database`.


In [5]:
# 03_SECRETS_AND_AUTH
from __future__ import annotations

import json
import os
from pathlib import Path

MARKER = "03_SECRETS_AND_AUTH"
print(MARKER)

# Hugging Face token — Phase 25: read ONLY from secrets/HF_TOKEN.json.
# No Colab Secrets fallback, no getpass. Format: {"HF_TOKEN": "hf_..."}.
hf_token_path = SECRETS_DIR / "HF_TOKEN.json"
if not hf_token_path.is_file():
    raise FileNotFoundError(
        f"HF_TOKEN.json not found at {hf_token_path}. "
        "Put a file with content {\"HF_TOKEN\": \"hf_...\"} there."
    )
try:
    _payload = json.loads(hf_token_path.read_text(encoding="utf-8"))
except Exception as _exc:
    raise RuntimeError(
        f"HF_TOKEN.json present but unreadable ({type(_exc).__name__}: {_exc})"
    )
_tok = (_payload or {}).get("HF_TOKEN")
if not _tok:
    raise RuntimeError(
        f"HF_TOKEN.json at {hf_token_path} has no \"HF_TOKEN\" key. "
        "Expected: {\"HF_TOKEN\": \"hf_...\"}."
    )
os.environ["HF_TOKEN"] = _tok
print(f"HF_TOKEN loaded from {hf_token_path}")

print("HF_TOKEN_SET:", bool(os.environ.get("HF_TOKEN")))

if os.environ.get("HF_TOKEN"):
    try:
        from huggingface_hub import HfApi, login
        login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
        who = HfApi(token=os.environ["HF_TOKEN"]).whoami()
        print("HF_AUTH_OK user:", who.get("name") or who.get("email") or "<ok>")
    except Exception as exc:
        print("HF_AUTH_FAIL", type(exc).__name__, str(exc)[:300])

# BigQuery SA path only; do not print key contents.
print("BQ_SA_PATH:", BQ_SA_PATH)
print("BQ_SA_EXISTS:", BQ_SA_PATH.exists())
if BQ_SA_PATH.exists():
    try:
        meta = json.loads(BQ_SA_PATH.read_text(encoding="utf-8"))
        print("BQ_SA_PROJECT:", meta.get("project_id"))
        print("BQ_SA_EMAIL:", meta.get("client_email"))
    except Exception as exc:
        print("BQ_SA_READ_FAIL", type(exc).__name__, str(exc)[:300])

# Snowflake secret path only; do not print password.
print("SNOWFLAKE_SECRET_JSON:", SNOWFLAKE_SECRET_JSON)
print("SNOWFLAKE_SECRET_EXISTS:", SNOWFLAKE_SECRET_JSON.exists())
if SNOWFLAKE_SECRET_JSON.exists():
    try:
        sf = json.loads(SNOWFLAKE_SECRET_JSON.read_text(encoding="utf-8"))
        print("SNOWFLAKE_ACCOUNT:", str(sf.get("account", ""))[:4] + "****")
        print("SNOWFLAKE_USER:", str(sf.get("user", ""))[:3] + "****")
        print("SNOWFLAKE_ROLE:", sf.get("role"))
        print("SNOWFLAKE_WAREHOUSE:", sf.get("warehouse"))
        print("SNOWFLAKE_DATABASE:", sf.get("database"))
    except Exception as exc:
        print("SNOWFLAKE_SECRET_READ_FAIL", type(exc).__name__, str(exc)[:300])


03_SECRETS_AND_AUTH
HF_TOKEN loaded from /content/drive/MyDrive/diploma_plan_sql/secrets/HF_TOKEN.json
HF_TOKEN_SET: True


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF_AUTH_OK user: DenisShubin
BQ_SA_PATH: /content/drive/MyDrive/diploma_plan_sql/secrets/spider2_bq_sa.json
BQ_SA_EXISTS: True
BQ_SA_PROJECT: project-0e0fc8a5-27b1-4e00-912
BQ_SA_EMAIL: denis-337@project-0e0fc8a5-27b1-4e00-912.iam.gserviceaccount.com
SNOWFLAKE_SECRET_JSON: /content/drive/MyDrive/diploma_plan_sql/secrets/snowflake.json
SNOWFLAKE_SECRET_EXISTS: True
SNOWFLAKE_ACCOUNT: RSRS****
SNOWFLAKE_USER: QOB****
SNOWFLAKE_ROLE: PARTICIPANT
SNOWFLAKE_WAREHOUSE: COMPUTE_WH_PARTICIPANT
SNOWFLAKE_DATABASE: PATENTS


## 04 — BigQuery probe

Проверяет BigQuery credentials, dry-run и короткий live query с billing cap. Секреты не печатает.


In [6]:
# 04_BIGQUERY_PROBE
from __future__ import annotations

import json
from pathlib import Path

MARKER = "04_BIGQUERY_PROBE"
print(MARKER)

BQ_READY = False
BQ_STATUS = {"ready": False, "reason": None}

try:
    from google.cloud import bigquery
    from google.oauth2 import service_account

    if not BQ_SA_PATH.exists():
        raise FileNotFoundError(f"BQ service account file not found: {BQ_SA_PATH}")

    creds = service_account.Credentials.from_service_account_file(str(BQ_SA_PATH))
    project = creds.project_id
    client = bigquery.Client(project=project, credentials=creds)
    print("BQ_CLIENT_OK project:", client.project)

    # Free dry run.
    dry = client.query("SELECT 1", job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    print("BQ_DRY_RUN_OK estimated_bytes:", dry.total_bytes_processed)

    # Tiny live query with hard cap.
    cfg = bigquery.QueryJobConfig(maximum_bytes_billed=50 * 1024 * 1024)
    job = client.query("SELECT COUNT(*) AS n FROM `bigquery-public-data.samples.shakespeare`", job_config=cfg)
    rows = list(job.result(timeout=60))
    print("BQ_LIVE_QUERY_OK rows:", rows[:1], "bytes_billed:", job.total_bytes_billed)

    BQ_READY = True
    BQ_STATUS = {"ready": True, "project": client.project, "bytes_billed": int(job.total_bytes_billed or 0)}
except Exception as exc:
    print("BQ_PROBE_FAIL", type(exc).__name__, str(exc)[:500])
    BQ_STATUS = {"ready": False, "reason": f"{type(exc).__name__}: {str(exc)[:300]}"}

print("BQ_READY:", BQ_READY)


04_BIGQUERY_PROBE
BQ_CLIENT_OK project: project-0e0fc8a5-27b1-4e00-912
BQ_DRY_RUN_OK estimated_bytes: 0
BQ_LIVE_QUERY_OK rows: [Row((164656,), {'n': 0})] bytes_billed: 0
BQ_READY: True


## 05 — Snowflake probe

Проверяет Snowflake connection, warehouse, current role/database и видимость баз. Если Snowflake пока не нужен, можно поставить `REQUIRE_SNOWFLAKE = False` в config.


In [7]:
# 05_SNOWFLAKE_PROBE
from __future__ import annotations

import json
import os

MARKER = "05_SNOWFLAKE_PROBE"
print(MARKER)

SNOWFLAKE_READY = False
SNOWFLAKE_STATUS = {"ready": False, "reason": None}

def _load_snowflake_cfg():
    cfg = {}
    if SNOWFLAKE_SECRET_JSON.exists():
        cfg.update(json.loads(SNOWFLAKE_SECRET_JSON.read_text(encoding="utf-8")))
    # Env vars override JSON.
    env_map = {
        "account": "SNOWFLAKE_ACCOUNT",
        "user": "SNOWFLAKE_USER",
        "password": "SNOWFLAKE_PASSWORD",
        "role": "SNOWFLAKE_ROLE",
        "warehouse": "SNOWFLAKE_WAREHOUSE",
        "database": "SNOWFLAKE_DATABASE",
    }
    for k, env in env_map.items():
        if os.environ.get(env):
            cfg[k] = os.environ[env]
    return cfg

try:
    import snowflake.connector

    cfg = _load_snowflake_cfg()
    required = ["account", "user", "password"]
    missing = [k for k in required if not cfg.get(k)]
    if missing:
        raise RuntimeError(f"Snowflake credentials missing keys: {missing}; use JSON or env vars")

    conn = snowflake.connector.connect(
        account=cfg["account"],
        user=cfg["user"],
        password=cfg["password"],
        role=cfg.get("role"),
        warehouse=cfg.get("warehouse"),
        database=cfg.get("database"),
        client_session_keep_alive=True,
    )
    cur = conn.cursor()
    cur.execute("SELECT CURRENT_ACCOUNT(), CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE(), CURRENT_VERSION()")
    row = cur.fetchone()
    print("SNOWFLAKE_SESSION_OK account/user/role/wh/db/version:", row)

    cur.execute("SHOW DATABASES")
    dbs = [r[1] for r in cur.fetchall()]
    print("SNOWFLAKE_DATABASES_VISIBLE:", len(dbs), "sample:", dbs[:10])
    cur.close()
    conn.close()

    SNOWFLAKE_READY = True
    SNOWFLAKE_STATUS = {"ready": True, "visible_databases": len(dbs), "sample": dbs[:10]}
except Exception as exc:
    print("SNOWFLAKE_PROBE_FAIL", type(exc).__name__, str(exc)[:500])
    SNOWFLAKE_STATUS = {"ready": False, "reason": f"{type(exc).__name__}: {str(exc)[:300]}"}

print("SNOWFLAKE_READY:", SNOWFLAKE_READY)


05_SNOWFLAKE_PROBE
SNOWFLAKE_SESSION_OK account/user/role/wh/db/version: ('SDB71929', 'QOBJWEZ_CX89548', 'PARTICIPANT', 'COMPUTE_WH_PARTICIPANT', 'PATENTS', '10.16.101')
SNOWFLAKE_DATABASES_VISIBLE: 159 sample: ['ADMIN_DB', 'ADVENTUREWORKS', 'AIRLINES', 'AMAZON_VENDOR_ANALYTICS__SAMPLE_DATASET', 'AUSTIN', 'BANK_SALES_TRADING', 'BASEBALL', 'BBC', 'BLS', 'BOWLINGLEAGUE']
SNOWFLAKE_READY: True


## 06 — GPU and model cache probe

Проверяет GPU, CUDA, PyTorch и доступность HF cache. Не грузит тяжёлую модель.


In [8]:
# 06_GPU_AND_MODEL_CACHE_PROBE
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

MARKER = "06_GPU_AND_MODEL_CACHE_PROBE"
print(MARKER)

GPU_READY = False
GPU_STATUS = {"ready": False, "reason": None}

try:
    import torch
    print("PYTHON:", sys.version.split()[0])
    print("TORCH:", torch.__version__)
    print("CUDA_AVAILABLE:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA_DEVICE_COUNT:", torch.cuda.device_count())
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f"GPU[{i}] name={props.name} total_gb={props.total_memory/1024**3:.1f}")
        GPU_READY = True
        GPU_STATUS = {"ready": True, "device_count": torch.cuda.device_count(), "name": torch.cuda.get_device_name(0)}
    else:
        GPU_STATUS = {"ready": False, "reason": "torch.cuda.is_available() is False"}
except Exception as exc:
    print("TORCH_GPU_PROBE_FAIL", type(exc).__name__, str(exc)[:300])
    GPU_STATUS = {"ready": False, "reason": f"{type(exc).__name__}: {str(exc)[:300]}"}

try:
    out = subprocess.run(["nvidia-smi"], text=True, capture_output=True, timeout=30)
    print("NVIDIA_SMI_RC:", out.returncode)
    print(out.stdout[:2000])
except Exception as exc:
    print("NVIDIA_SMI_WARN", type(exc).__name__, str(exc)[:300])

hf_home = Path(os.environ.get("HF_HOME", str(Path.home() / ".cache" / "huggingface")))
print("HF_HOME:", hf_home)
print("HF_HOME_EXISTS:", hf_home.exists())
print("GPU_READY:", GPU_READY)


06_GPU_AND_MODEL_CACHE_PROBE
PYTHON: 3.12.13
TORCH: 2.10.0+cu128
CUDA_AVAILABLE: True
CUDA_DEVICE_COUNT: 1
GPU[0] name=NVIDIA A100-SXM4-80GB total_gb=79.3
NVIDIA_SMI_RC: 0
Mon May 11 20:26:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             58W /  400W |       6MiB /  81

## 07 — Agent bridge setup

Запускает Flask server внутри Colab kernel и открывает его через Cloudflare quick tunnel. Агент будет обращаться к runtime через `tools/exec_remote.py`, без SendKeys и без фокуса VS Code.

После запуска скопируй `BRIDGE_URL` в локальный `tools/.bridge_url`, если агент сам его не синхронизирует.


In [9]:
# 07_AGENT_BRIDGE_SETUP_FIXED
# Robust bridge setup for Colab/VS Code.
# Safe to rerun. Reuses a healthy tunnel; otherwise restarts cloudflared.
from __future__ import annotations

import contextlib
import io
import os
import re
import subprocess
import sys
import threading
import time
import traceback
import urllib.request
from pathlib import Path

MARKER = "07_AGENT_BRIDGE_SETUP_FIXED"
print(MARKER)

# ---- prerequisites ----
for pkg in ["flask", "requests"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

from flask import Flask, jsonify, request, send_file
import requests

# These variables should be defined in 00/01 cells.
assert "PROJECT_ROOT" in globals(), "Run 00_SESSION_CONFIG and 01_MOUNT_DRIVE_AND_DIRS first"
assert "BRIDGE_URL_FILE" in globals(), "Run 00_SESSION_CONFIG first"

PROJECT_ROOT = Path(PROJECT_ROOT)
BRIDGE_URL_FILE = Path(BRIDGE_URL_FILE)

CF_BIN = Path("/content/cloudflared")
if not CF_BIN.exists():
    print("Downloading cloudflared...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        str(CF_BIN),
    )
    CF_BIN.chmod(0o755)
    print("CLOUDFLARED_READY", CF_BIN, CF_BIN.stat().st_size, "bytes")

PORT = int(os.environ.get("AGENT_BRIDGE_PORT", "5000"))

# ---- shared globals for remote exec ----
_SHARED_GLOBALS = globals().setdefault("_BRIDGE_SHARED_GLOBALS", {"__name__": "bridge_remote"})

# Keep important notebook variables visible to /exec code.
_SHARED_GLOBALS.update({
    "PROJECT_ROOT": PROJECT_ROOT,
    "BRIDGE_URL_FILE": BRIDGE_URL_FILE,
})

# ---- Flask app: create once, reuse on reruns ----
if "BRIDGE_APP" not in globals():
    BRIDGE_APP = Flask("agent_bridge")

    @BRIDGE_APP.route("/health")
    def health():
        return jsonify({
            "ok": True,
            "pid": os.getpid(),
            "project_root": str(PROJECT_ROOT),
        })

    @BRIDGE_APP.route("/exec", methods=["POST"])
    def execute_remote():
        payload = request.get_json(force=True, silent=True) or {}
        code = payload.get("code", "")
        if not code:
            return jsonify({"ok": False, "error": "no code"}), 400

        out_buf, err_buf = io.StringIO(), io.StringIO()

        try:
            with contextlib.redirect_stdout(out_buf), contextlib.redirect_stderr(err_buf):
                exec(code, _SHARED_GLOBALS)

            return jsonify({
                "ok": True,
                "stdout": out_buf.getvalue(),
                "stderr": err_buf.getvalue(),
            })

        except Exception:
            return jsonify({
                "ok": False,
                "stdout": out_buf.getvalue(),
                "stderr": err_buf.getvalue(),
                "traceback": traceback.format_exc(),
            }), 500

    @BRIDGE_APP.route("/file")
    def get_file():
        path = request.args.get("path", "")
        p = Path(path)

        if not path or not p.exists() or not p.is_file():
            return jsonify({
                "ok": False,
                "error": "not_found",
                "path": path,
            }), 404

        return send_file(str(p), as_attachment=True, download_name=p.name)

    @BRIDGE_APP.route("/ls")
    def ls():
        path = request.args.get("path", "/content")
        p = Path(path)

        if not p.exists():
            return jsonify({
                "ok": False,
                "error": "not_found",
                "path": path,
            }), 404

        items = []

        if p.is_dir():
            for x in sorted(p.iterdir()):
                items.append({
                    "name": x.name,
                    "type": "dir" if x.is_dir() else "file",
                    "size": x.stat().st_size if x.is_file() else None,
                })
        else:
            items.append({
                "name": p.name,
                "type": "file",
                "size": p.stat().st_size,
            })

        return jsonify({
            "ok": True,
            "path": str(p),
            "items": items,
        })


def _serve():
    BRIDGE_APP.run(
        host="127.0.0.1",
        port=PORT,
        debug=False,
        use_reloader=False,
        threaded=True,
    )


# ---- start/reuse local Flask ----
if "BRIDGE_THREAD" not in globals() or not globals()["BRIDGE_THREAD"].is_alive():
    BRIDGE_THREAD = threading.Thread(
        target=_serve,
        daemon=True,
        name="bridge-flask",
    )
    BRIDGE_THREAD.start()
    time.sleep(2)
    print(f"FLASK_STARTED http://127.0.0.1:{PORT}")
else:
    print(f"FLASK_ALREADY_STARTED http://127.0.0.1:{PORT}")


# ---- local health check ----
try:
    r_local = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=5)
    print("LOCAL_HEALTH:", r_local.status_code, r_local.text[:300])

    if r_local.status_code != 200:
        raise RuntimeError(r_local.text[:500])

except Exception as exc:
    raise RuntimeError(f"Flask is not reachable on localhost:{PORT}: {exc!r}")


# PHASE26_PID_CHECK: reject tunnels that route to a foreign runtime's kernel.
def _remote_health(url: str, timeout: int = 10, verbose: bool = True) -> bool:
    try:
        r = requests.get(url.rstrip("/") + "/health", timeout=timeout)
        if verbose:
            print("REMOTE_HEALTH_PROBE:", url, r.status_code, r.text[:300])
        if r.status_code != 200:
            return False
        try:
            data = r.json()
            ok = bool(data.get("ok") is True)
        except Exception:
            return '"ok":true' in r.text.replace(" ", "").lower()
        if not ok:
            return False
        # Reject if the tunnel routes to a different runtime's kernel.
        remote_pid = data.get("pid")
        local_pid = os.getpid()
        if remote_pid is not None and int(remote_pid) != int(local_pid):
            if verbose:
                print(f"FOREIGN_TUNNEL: remote pid {remote_pid} != local {local_pid}; will force new tunnel")
            return False
        return True
    except Exception as exc:
        if verbose:
            print("REMOTE_HEALTH_PROBE_FAILED:", url, repr(exc))
        return False


def _wait_remote_health(url: str, total_timeout: int = 90, step: int = 5) -> bool:
    """
    Cloudflare quick tunnel URL may need 10-60 seconds before DNS/route is ready.
    This waits instead of failing immediately.
    """
    deadline = time.time() + total_timeout
    attempt = 0

    while time.time() < deadline:
        attempt += 1

        if _remote_health(url, timeout=10, verbose=True):
            return True

        print(f"REMOTE_HEALTH_WAIT attempt={attempt}, sleeping={step}s")
        time.sleep(step)

    return False


# ---- 1) Try existing URL first ----
bridge_ready = False
candidate_urls = []

if "BRIDGE_URL" in globals() and isinstance(BRIDGE_URL, str) and BRIDGE_URL.startswith("https://"):
    candidate_urls.append(BRIDGE_URL.strip())

if BRIDGE_URL_FILE.exists():
    old = BRIDGE_URL_FILE.read_text(encoding="utf-8").strip()
    if old.startswith("https://"):
        candidate_urls.append(old)

seen = set()
for old_url in candidate_urls:
    if old_url in seen:
        continue

    seen.add(old_url)

    if _remote_health(old_url, timeout=8):
        BRIDGE_URL = old_url
        BRIDGE_URL_FILE.write_text(BRIDGE_URL + "\n", encoding="utf-8")

        print("BRIDGE_REUSED:", BRIDGE_URL)
        print("BRIDGE_URL_FILE:", BRIDGE_URL_FILE)
        print("BRIDGE_READY")

        bridge_ready = True
        break


# ---- 2) If no healthy URL, restart cloudflared cleanly ----
if not bridge_ready:
    if "BRIDGE_PROC" in globals() and BRIDGE_PROC is not None and BRIDGE_PROC.poll() is None:
        print(f"TERMINATING_OLD_CLOUDFLARED pid={BRIDGE_PROC.pid}")
        BRIDGE_PROC.terminate()

        try:
            BRIDGE_PROC.wait(timeout=5)
        except subprocess.TimeoutExpired:
            BRIDGE_PROC.kill()

    # Kill orphan /content/cloudflared tunnel processes left from old reruns.
    subprocess.run("pkill -f '/content/cloudflared tunnel' || true", shell=True)

    cmd = [
        str(CF_BIN),
        "tunnel",
        "--url", f"http://127.0.0.1:{PORT}",
        "--no-autoupdate",
        "--protocol", "http2",
        "--loglevel", "info",
    ]

    print("STARTING_CLOUDFLARED:", " ".join(cmd))

    BRIDGE_PROC = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        text=True,
    )

    url = None
    captured = []
    deadline = time.time() + 75

    import select

    while time.time() < deadline and url is None:
        if BRIDGE_PROC.poll() is not None:
            break

        if BRIDGE_PROC.stdout is None:
            time.sleep(0.5)
            continue

        ready, _, _ = select.select([BRIDGE_PROC.stdout], [], [], 0.5)

        if not ready:
            continue

        line = BRIDGE_PROC.stdout.readline()

        if not line:
            continue

        line = line.rstrip()
        captured.append(line)
        print(line)

        m = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0)
            break

    if not url:
        blob = "\n".join(captured)
        m = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", blob)
        if m:
            url = m.group(0)

    if not url:
        print("---- cloudflared captured tail ----")
        for line in captured[-40:]:
            print(line)

        raise RuntimeError(
            "Could not detect Cloudflare tunnel URL within 75s. "
            "Local Flask is healthy, but cloudflared did not publish a quick-tunnel URL. "
            "Re-run this cell; if it repeats, switch runtime network or use a named tunnel/ngrok fallback."
        )

    BRIDGE_URL = url
    BRIDGE_URL_FILE.write_text(BRIDGE_URL + "\n", encoding="utf-8")

    print("BRIDGE_URL:", BRIDGE_URL)
    print("BRIDGE_URL_FILE:", BRIDGE_URL_FILE)

    if not _wait_remote_health(BRIDGE_URL, total_timeout=90, step=5):
        raise RuntimeError(
            f"Cloudflare URL was created but /health is not reachable after waiting: {BRIDGE_URL}"
        )

    print("BRIDGE_READY")


# ---- final shared globals refresh ----
_SHARED_GLOBALS.update({
    "PROJECT_ROOT": PROJECT_ROOT,
    "BRIDGE_URL_FILE": BRIDGE_URL_FILE,
    "BRIDGE_URL": BRIDGE_URL,
})

print("FINAL_BRIDGE_URL:", BRIDGE_URL)
print("FINAL_BRIDGE_URL_FILE:", BRIDGE_URL_FILE)
print("DONE")

# Phase 25: mirror BRIDGE_URL to <PROJECT_ROOT>/tools/.bridge_url
# (the local Windows repo Drive-syncs this path; tools/exec_remote.py
#  reads from there).
TOOLS_BRIDGE_URL = PROJECT_ROOT / "tools" / ".bridge_url"
TOOLS_BRIDGE_URL.parent.mkdir(parents=True, exist_ok=True)
TOOLS_BRIDGE_URL.write_text(BRIDGE_URL + "\n", encoding="utf-8")
print("TOOLS_BRIDGE_URL:", TOOLS_BRIDGE_URL)


07_AGENT_BRIDGE_SETUP_FIXED
CLOUDFLARED_READY /content/cloudflared 39667364 bytes
 * Serving Flask app 'agent_bridge'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:26:55] "GET /health HTTP/1.1" 200 -


FLASK_STARTED http://127.0.0.1:5000
LOCAL_HEALTH: 200 {"ok":true,"pid":1524,"project_root":"/content/drive/MyDrive/diploma_plan_sql"}

REMOTE_HEALTH_PROBE_FAILED: https://electoral-spirit-lace-art.trycloudflare.com ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'electoral-spirit-lace-art.trycloudflare.com\', port=443): Max retries exceeded with url: /health (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7fcf7807c4d0>: Failed to resolve \'electoral-spirit-lace-art.trycloudflare.com\' ([Errno -2] Name or service not known)"))'))
STARTING_CLOUDFLARED: /content/cloudflared tunnel --url http://127.0.0.1:5000 --no-autoupdate --protocol http2 --loglevel info
2026-05-11T20:26:55Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www

INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:27:31] "GET /health HTTP/1.1" 200 -


REMOTE_HEALTH_PROBE: https://toronto-decreased-participation-retention.trycloudflare.com 200 {"ok":true,"pid":1524,"project_root":"/content/drive/MyDrive/diploma_plan_sql"}

BRIDGE_READY
FINAL_BRIDGE_URL: https://toronto-decreased-participation-retention.trycloudflare.com
FINAL_BRIDGE_URL_FILE: /content/drive/MyDrive/diploma_plan_sql/.bridge_url
DONE
TOOLS_BRIDGE_URL: /content/drive/MyDrive/diploma_plan_sql/tools/.bridge_url


## 08 — Full agent readiness check

Финальная ячейка перед передачей работы агенту. Она собирает всё в один JSON/Markdown отчёт и явно говорит: `READY_FOR_AGENT = True/False`.

Запускай её после всех предыдущих ячеек, особенно после bridge.


In [ ]:
# 08_FULL_AGENT_READINESS_CHECK
from __future__ import annotations

import datetime as dt
import json
import os
import platform
import socket
import subprocess
import sys
from pathlib import Path

MARKER = "08_FULL_AGENT_READINESS_CHECK"
print(MARKER)

checks = []

def add_check(name, ok, required=True, details=None, fix=None):
    row = {
        "name": name,
        "ok": bool(ok),
        "required": bool(required),
        "details": details or {},
        "fix": fix or "",
    }
    checks.append(row)
    flag = "OK" if ok else ("FAIL" if required else "WARN")
    print(f"[{flag}] {name}: {details if details is not None else ''}")

# 1. Drive / paths
add_check("project_root_exists", PROJECT_ROOT.exists(), True, {"path": str(PROJECT_ROOT)})
try:
    p = OUTPUTS_DIR / "logs" / "readiness_write_probe.txt"
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(dt.datetime.now(dt.timezone.utc).isoformat() + "\n", encoding="utf-8")
    add_check("drive_write_ok", p.exists() and p.stat().st_size > 0, True, {"path": str(p), "size": p.stat().st_size})
except Exception as exc:
    add_check("drive_write_ok", False, True, {"error": f"{type(exc).__name__}: {str(exc)[:300]}"}, "Re-mount Drive and check PROJECT_ROOT")

# 2. Python / packages
add_check("python_version", sys.version_info >= (3, 10), True, {"python": sys.version.split()[0]})
for mod in ["flask", "requests", "huggingface_hub", "google.cloud.bigquery", "snowflake.connector", "sqlglot", "jsonschema", "func_timeout"]:
    try:
        __import__(mod.split('.')[0])
        add_check(f"import_{mod}", True, True)
    except Exception as exc:
        add_check(f"import_{mod}", False, True, {"error": str(exc)[:200]}, "Run 02_INSTALL_CORE_PACKAGES")

# 3. HF token
add_check("hf_token_set", bool(os.environ.get("HF_TOKEN")), REQUIRE_HF_TOKEN, {"set": bool(os.environ.get("HF_TOKEN"))}, "Run 03_SECRETS_AND_AUTH")
if os.environ.get("HF_TOKEN"):
    try:
        from huggingface_hub import HfApi
        who = HfApi(token=os.environ["HF_TOKEN"]).whoami()
        add_check("hf_auth_ok", True, REQUIRE_HF_TOKEN, {"user": who.get("name") or "<ok>"})
    except Exception as exc:
        add_check("hf_auth_ok", False, REQUIRE_HF_TOKEN, {"error": f"{type(exc).__name__}: {str(exc)[:300]}"}, "Check HF token / gated model access")

# 4. GPU
try:
    import torch
    cuda = torch.cuda.is_available()
    details = {"cuda": cuda, "torch": torch.__version__}
    if cuda:
        details.update({"device_count": torch.cuda.device_count(), "device0": torch.cuda.get_device_name(0)})
    add_check("gpu_cuda_available", cuda, REQUIRE_GPU, details, "Runtime > Change runtime type > GPU")
except Exception as exc:
    add_check("gpu_cuda_available", False, REQUIRE_GPU, {"error": str(exc)[:300]}, "Install/repair torch or restart runtime")

# 5. BigQuery
if 'BQ_READY' in globals():
    add_check("bigquery_probe_ok", BQ_READY, REQUIRE_BQ, BQ_STATUS, "Run 04_BIGQUERY_PROBE / check service account")
else:
    add_check("bigquery_probe_ok", False, REQUIRE_BQ, {"reason": "BQ_READY not defined"}, "Run 04_BIGQUERY_PROBE")

# 6. Snowflake
if 'SNOWFLAKE_READY' in globals():
    add_check("snowflake_probe_ok", SNOWFLAKE_READY, REQUIRE_SNOWFLAKE, SNOWFLAKE_STATUS, "Run 05_SNOWFLAKE_PROBE / check credentials")
else:
    add_check("snowflake_probe_ok", False, REQUIRE_SNOWFLAKE, {"reason": "SNOWFLAKE_READY not defined"}, "Run 05_SNOWFLAKE_PROBE")

# 7. Spider2 data candidates
spider2_candidates = {
    "spider2_lite_jsonl_or_raw": [
        PROJECT_ROOT / "external_benchmarks" / "spider2_lite" / "processed" / "spider2_lite_547.jsonl",
        PROJECT_ROOT / "external_benchmarks" / "spider2_lite" / "raw" / "Spider2" / "spider2-lite" / "spider2-lite.jsonl",
        PROJECT_ROOT / "data" / "spider2_lite" / "spider2-lite.jsonl",
    ],
    "spider2_snow_jsonl": [
        PROJECT_ROOT / "external_benchmarks" / "spider2_snow" / "processed" / "spider2_snow_547.jsonl",
        PROJECT_ROOT / "data" / "spider2_snow" / "raw" / "spider2-snow.jsonl",
    ],
    "spider2_dbt_examples": [
        PROJECT_ROOT / "external_benchmarks" / "spider2_dbt" / "examples",
        PROJECT_ROOT / "data" / "spider2_dbt" / "examples",
    ],
}
for name, paths in spider2_candidates.items():
    existing = [str(p) for p in paths if p.exists()]
    add_check(name, bool(existing), REQUIRE_SPIDER2_DATA, {"existing": existing, "checked": [str(p) for p in paths]}, "Check Drive sync / benchmark acquisition")

# 8. Bridge
bridge_url = globals().get("BRIDGE_URL")
if not bridge_url and BRIDGE_URL_FILE.exists():
    bridge_url = BRIDGE_URL_FILE.read_text(encoding="utf-8").strip()
if bridge_url:
    try:
        import requests
        r = requests.get(bridge_url.rstrip("/") + "/health", timeout=20)
        add_check("bridge_health_ok", r.status_code == 200 and r.json().get("ok"), REQUIRE_BRIDGE, {"url": bridge_url, "status": r.status_code, "body": r.text[:200]}, "Re-run 07_AGENT_BRIDGE_SETUP")
    except Exception as exc:
        add_check("bridge_health_ok", False, REQUIRE_BRIDGE, {"url": bridge_url, "error": f"{type(exc).__name__}: {str(exc)[:300]}"}, "Re-run 07_AGENT_BRIDGE_SETUP")
else:
    add_check("bridge_health_ok", False, REQUIRE_BRIDGE, {"reason": "BRIDGE_URL not found"}, "Run 07_AGENT_BRIDGE_SETUP")

# 9. Optional DBT remote
if CHECK_DBT_REMOTE:
    try:
        cmd = ["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=10", DBT_REMOTE, f"cd {DBT_REMOTE_PROJECT} && pwd && . .venv/bin/activate && dbt --version | head -20"]
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        add_check("dbt_remote_ok", res.returncode == 0, False, {"rc": res.returncode, "stdout": res.stdout[-1000:], "stderr": res.stderr[-500:]}, "Check SSH key / DBT server")
    except Exception as exc:
        add_check("dbt_remote_ok", False, False, {"error": f"{type(exc).__name__}: {str(exc)[:300]}"}, "Check SSH key / DBT server")

required_failed = [c for c in checks if c["required"] and not c["ok"]]
READY_FOR_AGENT = len(required_failed) == 0

summary = {
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(),
    "ready_for_agent": READY_FOR_AGENT,
    "project_root": str(PROJECT_ROOT),
    "bridge_url": bridge_url,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "checks": checks,
    "required_failed": required_failed,
}

json_path = OUTPUTS_DIR / "logs" / "agent_readiness_check.json"
md_path = OUTPUTS_DIR / "logs" / "agent_readiness_check.md"
json_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

md_lines = [
    "# Agent readiness check",
    "",
    f"Checked at: {summary['ts']}",
    f"READY_FOR_AGENT: **{READY_FOR_AGENT}**",
    "",
    "| Check | Required | OK | Details | Fix |",
    "|---|---:|---:|---|---|",
]
for c in checks:
    details = json.dumps(c["details"], ensure_ascii=False)[:500]
    md_lines.append(f"| `{c['name']}` | {c['required']} | {c['ok']} | `{details}` | {c['fix']} |")
md_path.write_text("\n".join(md_lines) + "\n", encoding="utf-8")

print("\nREADY_FOR_AGENT:", READY_FOR_AGENT)
print("READINESS_JSON:", json_path)
print("READINESS_MD:", md_path)
if required_failed:
    print("\nBLOCKING FAILURES:")
    for c in required_failed:
        print("-", c["name"], "=>", c["fix"])
else:
    print("\nAll required infrastructure checks passed. You can give the agent the bridge URL and start the next task.")


08_FULL_AGENT_READINESS_CHECK
[OK] project_root_exists: {'path': '/content/drive/MyDrive/diploma_plan_sql'}
[OK] drive_write_ok: {'path': '/content/drive/MyDrive/diploma_plan_sql/outputs/logs/readiness_write_probe.txt', 'size': 33}
[OK] python_version: {'python': '3.12.13'}
[OK] import_flask: 
[OK] import_requests: 
[OK] import_huggingface_hub: 
[OK] import_google.cloud.bigquery: 
[OK] import_snowflake.connector: 
[OK] import_sqlglot: 
[OK] import_jsonschema: 
[OK] import_func_timeout: 
[OK] hf_token_set: {'set': True}
[OK] hf_auth_ok: {'user': 'DenisShubin'}
[OK] gpu_cuda_available: {'cuda': True, 'torch': '2.10.0+cu128', 'device_count': 1, 'device0': 'NVIDIA A100-SXM4-80GB'}
[OK] bigquery_probe_ok: {'ready': True, 'project': 'project-0e0fc8a5-27b1-4e00-912', 'bytes_billed': 0}
[OK] snowflake_probe_ok: {'ready': True, 'visible_databases': 159, 'sample': ['ADMIN_DB', 'ADVENTUREWORKS', 'AIRLINES', 'AMAZON_VENDOR_ANALYTICS__SAMPLE_DATASET', 'AUSTIN', 'BANK_SALES_TRADING', 'BASEBALL', 'BB

INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:28:45] "GET /health HTTP/1.1" 200 -


[OK] spider2_dbt_examples: {'existing': ['/content/drive/MyDrive/diploma_plan_sql/external_benchmarks/spider2_dbt/examples'], 'checked': ['/content/drive/MyDrive/diploma_plan_sql/external_benchmarks/spider2_dbt/examples', '/content/drive/MyDrive/diploma_plan_sql/data/spider2_dbt/examples']}
[OK] bridge_health_ok: {'url': 'https://toronto-decreased-participation-retention.trycloudflare.com', 'status': 200, 'body': '{"ok":true,"pid":1524,"project_root":"/content/drive/MyDrive/diploma_plan_sql"}\n'}

READY_FOR_AGENT: True
READINESS_JSON: /content/drive/MyDrive/diploma_plan_sql/outputs/logs/agent_readiness_check.json
READINESS_MD: /content/drive/MyDrive/diploma_plan_sql/outputs/logs/agent_readiness_check.md

All required infrastructure checks passed. You can give the agent the bridge URL and start the next task.


INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:30:25] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:30:27] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:30:27] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:30:43] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:30:51] "POST /exec HTTP/1.1" 200 -


LOAD_EMITTER qwen2_5_coder_7b ...


...

INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:36:02] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:36:25] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:36:30] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:36:45] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:36:50] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:36:54] "POST /exec HTTP/1.1" 200 -


[snow_full_v28_revert_a] catalog loaded: 572997 cols
[snow_full_v28_revert_a] catalog partitioned by db: 150 unique DBs (sizes: min=4, max=71832, avg=3819)
[snow_full_v28_revert_a] tasks selected: 547
[snow_full_v28_revert_a] RESUMED: 403 prior done (sv=274 parse=369 exec=92 plan=109), 403 skipped, 144 remaining


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:38:31] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:38:49] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:40:50] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:42:50] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:44:51] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:46:51] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:47:02] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:50:03] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:53:03] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:56:04] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 20:59:04] "POST /

[snow_full_v28_revert_a] DONE n=547 sv=383 parse=503 exec=130 guard_leaks=0 guard_rewrites=14


Exception in thread Phase28S1Supervisor:
Traceback (most recent call last):
  File "<string>", line 79, in _supervisor
TypeError: 'PosixPath' object is not callable

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "<string>", line 165, in _supervisor
TypeError: 'PosixPath' object is not callable
INFO:werkzeug:127.0.0.1 - - [11/May/2026 23:49:00] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 23:49:16] "POST /exec HTTP/1.1" 500 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 23:49:24] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 23:49:47] "POST /exec HTTP/1.1" 500 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 23:49:58] "POST /exec HTTP/1.1" 500 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 23:50:09

## 09 — Legacy notes

Исходный `example.ipynb` содержал старые B0/B1 smoke cells и ячейку с hardcoded BigQuery service account JSON. В этой версии оставлен только чистый infrastructure setup. Старые экспериментальные ячейки лучше запускать агентом через scripts/runners, а не руками из setup-notebook.


In [17]:
print(1)